# ADM & LT 2025/2026: Hackathon

This notebook implements a modular query-answering pipeline for the hackathon task. The system is designed to answer natural language questions using the structured textual evidence associated with each database.

The implementation combines query-type routing, hybrid retrieval, reranking, language-model-based extraction, and deterministic heuristics. The pipeline addresses the main query categories required by the task: Lookup, Boolean, Count, Aggregation, and Set operations.

The notebook also documents the LoRA adaptation procedure used for the Qwen component and then applies the resulting pipeline to validation and submission generation.

In [ ]:
!pip install polars rank_bm25 transformers peft accelerate scikit-learn sentence-transformers sentencepiece -q

In [ ]:
import os, zipfile, json, re
import numpy as np
import pandas as pd
import polars as pl
import torch
from tqdm.auto import tqdm
from collections import Counter

import json
import re
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from rank_bm25 import BM25Okapi

from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import T5ForConditionalGeneration, T5Tokenizer
from peft import PeftModel

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline as SKPipeline

In [ ]:
# Step 1: Extract compressed input files when present.
for root, dirs, files in os.walk("/kaggle/input"):
    for fname in files:
        if fname.endswith(".zip"):
            zp = os.path.join(root, fname)
            ex = os.path.join(root, fname.replace(".zip","_extracted"))
            if not os.path.exists(ex):
                print(f"Extracting {fname}...")
                with zipfile.ZipFile(zp,'r') as z: z.extractall(ex)

# Step 2: Locate dataset, evidence corpus, and LoRA adapter paths.
BASE_PATH = CORPUS_DIR = LORA_PATH = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "train.parquet" in files and BASE_PATH is None:
        BASE_PATH = root
    if "evidence_corpus" in dirs and CORPUS_DIR is None:
        CORPUS_DIR = os.path.join(root, "evidence_corpus")
    if os.path.basename(root) == "evidence_corpus" and any(f.endswith(".jsonl") for f in files) and CORPUS_DIR is None:
        CORPUS_DIR = root
    if "adapter_config.json" in files and LORA_PATH is None:
        LORA_PATH = root

print(f"Data:   {BASE_PATH}")
print(f"Corpus: {CORPUS_DIR}")
print(f"LoRA:   {LORA_PATH}")

train = pl.read_parquet(f"{BASE_PATH}/train.parquet").to_pandas()
dev   = pl.read_parquet(f"{BASE_PATH}/dev.parquet").to_pandas()
test  = pd.read_csv(f"{BASE_PATH}/test.csv")
print(f"Train: {len(train)} | Dev: {len(dev)} | Test: {len(test)}")

In [ ]:
test_evidence = {}
for fname in tqdm(os.listdir(CORPUS_DIR), desc="Loading corpus"):
    if fname.endswith(".jsonl"):
        db_id = fname.replace("db_","").replace(".jsonl","")
        facts = []
        with open(os.path.join(CORPUS_DIR, fname)) as f:
            for line in f:
                facts.append(json.loads(line)["text"])
        test_evidence[db_id] = facts
print(f"Loaded {len(test_evidence)} databases")

In [ ]:
# This cell documents and optionally reproduces the LoRA fine-tuning procedure for the Qwen component.
# In the inference pipeline, the trained adapter is loaded from the configured Kaggle input path unless this cell is executed and its output path is selected explicitly.

from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer

# 1. Prepare the Qwen fine-tuning dataset for Lookup, Count, and Set operation queries.
target_df = train[train["query_type"].isin(["Lookup", "Count", "Set ops"])].copy().reset_index(drop=True)

target_df = target_df.sample(n=10000, random_state=42)

train_samples = []
for _, row in tqdm(target_df.iterrows(), total=len(target_df), desc="Preparing Qwen LoRA Data"):
    query = row["query"]
    gold = str(row["gold_answer"]).strip()
    facts = [str(f) for f in row["db_facts"]]
    
    if len(gold) < 1: continue
    
    q_type = row["query_type"]
    
    top_k_train = 20 if q_type in ["Count", "Set ops"] else 8
    
    tokenized = [f.lower().split() for f in facts]
    bm25 = BM25Okapi(tokenized)
    bm25_scores = bm25.get_scores(query.lower().split())
    top_idx = np.argsort(bm25_scores)[::-1][:top_k_train]
    top_facts = [facts[i] for i in top_idx]
    
    context = "\n".join([f"- {f}" for f in top_facts])
    
    if q_type == "Set ops":
        sys_prompt = "You are a precise data extraction assistant. Reply with ONLY a valid JSON array of strings."
    else:
        sys_prompt = "You are a precise data extraction assistant. Reply with ONLY the exact entity name or the exact numerical count."
        
    messages = [
        {"role": "system", "content": sys_prompt},
        {"role": "user", "content": f"Facts:\n{context}\n\nQuestion: {query}"},
        {"role": "assistant", "content": gold}
    ]
    train_samples.append({"messages": messages})

train_dataset = Dataset.from_list(train_samples)

# 2. Load the base model for fine-tuning.
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def format_chat(example):
    example["text"] = tokenizer.apply_chat_template(example["messages"], tokenize=False)
    return example

train_dataset = train_dataset.map(format_chat)

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map="auto")
model.config.use_cache = False 

# 3. Configure the LoRA adaptation module.
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)

# 4. Train and save the adapted model using memory-aware settings.
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="/kaggle/working/qwen-lora-ft",
    num_train_epochs=2,
    per_device_train_batch_size=1,      
    gradient_accumulation_steps=16,      
    gradient_checkpointing=True,         
    learning_rate=2e-4,
    logging_steps=50,
    save_strategy="epoch",
    report_to="none",
    fp16=torch.cuda.is_available(),
    dataset_text_field="text"            
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    args=training_args
)

print("\nStarting Qwen LoRA fine-tuning with memory-aware settings...")
trainer.train()
trainer.model.save_pretrained("/kaggle/working/qwen-lora-ft/final")
tokenizer.save_pretrained("/kaggle/working/qwen-lora-ft/final")
print("Fine-tuning completed and adapter saved.")

In [ ]:
device   = "cuda" if torch.cuda.is_available() else "cpu"
hf_dev   = 0 if torch.cuda.is_available() else -1
dtype    = torch.float16 if torch.cuda.is_available() else torch.float32

print("Loading BGE-Large (335M)...")
retriever = SentenceTransformer("BAAI/bge-large-en-v1.5", device=device)

print("Loading MiniLM-L12 reranker (33M)...")
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-12-v2", device=device)

# FLAN-T5 component for Boolean and Aggregation queries.
print("Loading FLAN-T5-base (250M)...")
flan_tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-base")
flan_model = T5ForConditionalGeneration.from_pretrained(
    "google/flan-t5-base", torch_dtype=dtype).to(device)

# Qwen component with LoRA adapter for Lookup, Count, and Set operation queries.
print("Loading Qwen2.5-0.5B-Instruct + LoRA...")
BASE_MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
base_llm = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID, torch_dtype=torch.float16, device_map="auto")
if LORA_PATH:
    llm = PeftModel.from_pretrained(base_llm, LORA_PATH)
    print("LoRA adapter loaded.")
else:
    llm = base_llm
    print("Base Qwen model loaded without a LoRA adapter.")
llm.eval()

print("
Generation components loaded successfully.")
print("Retrieval components loaded successfully.")

In [ ]:
# Query-type router trained with TF-IDF features and Logistic Regression.
router_model = SKPipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), max_features=5000)),
    ("clf",   LogisticRegression(max_iter=1000, C=5))
])
router_model.fit(train["query"], train["query_type"])
print(f"Router trained on {len(train)} queries")

def router(query):
    return router_model.predict([query])[0]

In [ ]:
# Utility functions for normalization, parsing, and Boolean answer handling.

def normalize_space(text):
    return re.sub(r"\s+", " ", str(text)).strip()

def clean_answer(ans):
    ans = normalize_space(ans)
    ans = re.sub(r"^(answer\s*:|the answer is\s*)", "", ans, flags=re.I).strip()
    return ans.strip(' \n\t\"\'.` .,;:')

def has_word(text, words):
    text = str(text).lower()
    return any(re.search(rf"\b{re.escape(w.lower())}\b", text) for w in words)

def parse_yes_no(ans):
    ans = clean_answer(ans).lower()
    if re.match(r"^(yes|true)\b", ans): return "yes"
    if re.match(r"^(no|false)\b", ans): return "no"
    if "yes" in ans and "no" not in ans: return "yes"
    if "no" in ans and "yes" not in ans: return "no"
    return None

def parse_single_integer(ans, max_value=1000):
    ans = clean_answer(ans)
    match = re.fullmatch(r"\D*(\d{1,6})\D*", ans)
    if not match: return None
    value = int(match.group(1))
    return str(value) if 0 <= value <= max_value else None

def asks_female(query):
    return has_word(query, ["female","woman","women"])

def asks_male(query):
    return has_word(query, ["male","man","men"]) and not asks_female(query)

FEMALE_WORDS = ["she","her","hers","female","woman","women","actress"]
MALE_WORDS   = ["he","him","his","male","man","men","actor"]

def fact_has_female_marker(fact):
    return has_word(fact, FEMALE_WORDS)

def fact_has_male_marker(fact):
    return has_word(fact, MALE_WORDS) and not fact_has_female_marker(fact)

def flan_extract(prompt, max_new_tokens=100):
    inputs = flan_tokenizer(
        prompt, return_tensors="pt", max_length=512, truncation=True).to(device)
    with torch.no_grad():
        outputs = flan_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return flan_tokenizer.decode(outputs[0], skip_special_tokens=True)

# Utility functions for entity extraction and fact access.

def get_person_name(fact):
    stopwords = {"the","a","an","it","he","she","they","this","that",
                 "its","his","her","their","born","in","at","of","by","if","all"}
    parts = []
    for word in fact.split()[:5]:
        clean = word.strip('.,;:!?()[]{}"- ')
        if len(clean) < 2: break
        if clean[0].isupper() and clean.lower() not in stopwords:
            parts.append(clean)
        else: break
    return " ".join(parts) if len(parts) >= 2 else None

def is_person_fact(fact):
    fl = fact.lower()
    return any(p in fl for p in [
        "was born","is born","is a ","is an ","was a ","was an ",
        "graduated from","studied at","worked at","works at",
        "played for","plays for"," she "," her "," he "," his ",
        "a human","human being","married","actor","musician","player"])

def get_facts(row, is_test=False):
    if is_test: return test_evidence[str(row["db_id"])]
    return [str(f) for f in row["db_facts"]]

# Hybrid retrieval with lexical search, dense embeddings, and cross-encoder reranking.

def hybrid_retrieve_reranked(query, facts, current_db_id=None, top_k=8, rerank_pool=30):
    if len(facts) <= top_k: return facts
    tokenized = [f.lower().split() for f in facts]
    bm25 = BM25Okapi(tokenized)
    bm25_scores = bm25.get_scores(query.lower().split())
    bm25_ranks  = {i: r for r, i in enumerate(np.argsort(bm25_scores)[::-1])}
    q_emb = retriever.encode(
        [f"Represent this sentence for searching relevant passages: {query}"],
        convert_to_numpy=True, normalize_embeddings=True)
    db_key = str(current_db_id) if current_db_id else None
    if db_key and db_key in corpus_embeddings:
        fact_embs = corpus_embeddings[db_key]
    else:
        fact_embs = retriever.encode(facts, convert_to_numpy=True,
                                      normalize_embeddings=True, show_progress_bar=False)
    dense_scores = np.dot(fact_embs, q_emb.T).squeeze()
    if dense_scores.ndim == 0: dense_scores = np.array([dense_scores])
    dense_ranks  = {i: r for r, i in enumerate(np.argsort(dense_scores)[::-1])}
    k = 60
    rrf = {i: 1/(k+bm25_ranks.get(i,len(facts))) + 1/(k+dense_ranks.get(i,len(facts)))
           for i in range(len(facts))}
    pool = sorted(rrf, key=rrf.get, reverse=True)[:rerank_pool]
    candidates = [facts[i] for i in pool]
    scores = reranker.predict([[query, f] for f in candidates])
    return [candidates[i] for i in np.argsort(scores)[::-1][:top_k]]

def qwen_generate(prompt, max_new_tokens=50,
                  system_prompt="You are a precise data extraction assistant."):
    messages = [{"role":"system","content":system_prompt},
                {"role":"user","content":prompt}]
    text   = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=1500).to(llm.device)
    with torch.no_grad():
        outputs = llm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                                pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

print("Utility functions OK")

In [ ]:
# Query-type-specific answer functions

def get_person_name_clean(fact):
    stopwords = {"the","a","an","it","he","she","they","this","that",
                 "its","his","her","their","born","in","at","of","by","if","all"}
    parts = []
    for word in fact.split()[:5]:
        clean = word.strip('.,;:!?()[]{}"- ')
        if len(clean) < 2: break
        if clean[0].isupper() and clean.lower() not in stopwords: 
            parts.append(clean)
        else: break
    return " ".join(parts) if len(parts) >= 2 else None

# Lookup questions: Qwen component with LoRA adapter.
def answer_lookup(query, facts, current_db_id=None):
    top_facts = hybrid_retrieve_reranked(query, facts, current_db_id, top_k=8, rerank_pool=40)
    context = "\n".join([f"- {f}" for f in top_facts])
    ans = qwen_generate(
        f"Facts:\n{context}\n\nQuestion: {query}", 
        max_new_tokens=15,
        system_prompt="You are a precise data extraction assistant. Reply with ONLY the exact entity name or the exact numerical count."
    )
    return re.sub(r"[^\w\s\-']", "", ans).strip()

def answer_count(query, facts, current_db_id=None):
    query_lower = query.lower()
    
    # A. Direct heuristic for population and capacity-related count questions.
    if any(w in query_lower for w in ["citizen","inhabitant","population","national","resident","people in","capacity"]):
        tokenized = [f.lower().split() for f in facts]
        bm25 = BM25Okapi(tokenized)
        scores = bm25.get_scores(query_lower.split())
        for i in np.argsort(scores)[::-1][:5]:
            if scores[i] > 0:
                for n in re.findall(r'\b(\d[\d,]*)\b', facts[i]):
                    val = int(n.replace(',',''))
                    if val > 500 and not (1800 <= val <= 2030): 
                        return str(val)

    # B. Gender-based entity counting heuristic.
    if "female" in query_lower or ("woman" in query_lower and "male" not in query_lower):
        people = set()
        for fact in facts:
            if any(w in fact.lower() for w in [" she ", " her ", " female ", " woman ", " actress ", " sister "]):
                name = get_person_name_clean(fact)
                if name: people.add(name)
        if people: return str(len(people))

    if ("male" in query_lower or " man " in query_lower) and "female" not in query_lower:
        people = set()
        for fact in facts:
            fl = fact.lower()
            if any(w in fl for w in [" she ", " her ", " female ", " woman ", " actress ", " sister "]): continue
            if any(w in fl for w in [" he ", " his ", " male ", " man ", " actor ", " brother "]):
                name = get_person_name_clean(fact)
                if name: people.add(name)
        if people: return str(len(people))

    # C. Contextual Qwen fallback for remaining count questions.
    top_facts = hybrid_retrieve_reranked(query, facts, current_db_id, top_k=10, rerank_pool=40)
    context = "\n".join([f"- {f}" for f in top_facts])
    ans = qwen_generate(
        f"Facts:\n{context}\n\nQuestion: {query}", 
        max_new_tokens=10,
        system_prompt="You are a precise data extraction assistant. Reply with ONLY the exact entity name or the exact numerical count."
    )
    nums = re.findall(r"\d+", ans)
    return nums[0] if nums else "0"

def parse_number_from_fact(fact):
    """
    Extract the largest numeric value from a fact.
    The procedure handles plus signs, comma-separated numbers, and decimal values.
    """
    cleaned = re.sub(r'[+]', '', fact)          # Remove plus signs.
    cleaned = re.sub(r'(\d),(\d)', r'\1\2', cleaned)  # Remove commas within numeric expressions.
    nums = re.findall(r'\b(\d+(?:\.\d+)?)\b', cleaned)
    values = []
    for n in nums:
        try:
            val = float(n)
            if val >= 1:
                values.append(val)
        except:
            pass
    return max(values) if values else None


def extract_numeric_candidates(facts, numeric_kw, query_lower):
    """
    Scan all facts to identify numeric candidates.
    A fact is considered only when it contains at least one numeric value.
    """
    # Synonyms used to broaden lexical matching.
    SYNONYMS = {
        "crowd":       ["attendance", "attendees", "people", "audience", "spectators"],
        "attendance":  ["crowd", "attendees", "people", "audience", "spectators", "visitors"],
        "voters":      ["eligible", "registered", "electorate"],
        "production":  ["produced", "built", "manufactured", "units", "made"],
        "population":  ["inhabitants", "residents", "citizens", "people"],
        "unemployment":["jobless", "unemployed"],
    }

    # Expand query tokens with the synonym dictionary.
    query_tokens = set(re.findall(r'\b\w+\b', query_lower)) - {
        "what", "is", "was", "the", "a", "an", "of", "in", "how", "many",
        "largest", "smallest", "highest", "lowest", "most", "least",
        "biggest", "fewest", "best", "worst", "oldest", "youngest",
        "maximum", "minimum", "number", "which", "who", "much", "run"
    }
    expanded_tokens = set(query_tokens)
    for tok in query_tokens:
        if tok in SYNONYMS:
            expanded_tokens.update(SYNONYMS[tok])

    candidates = []
    for fact in facts:
        fl = fact.lower()

        # Semantic filter: at least one expanded token must appear in the fact.
        # If the query contains no useful tokens, all facts are retained.
        if expanded_tokens and not any(tok in fl for tok in expanded_tokens if len(tok) > 2):
            continue

        val = parse_number_from_fact(fact)
        if val is None:
            continue

        subj = get_person_name_clean(fact) or \
               fact.split(" has ")[0].split(" is ")[0] \
                   .split(" of ")[0].split(" for ")[0] \
                   .split(" produced")[0].split(" built")[0].strip()

        candidates.append((val, subj, fact))

    return candidates


def answer_aggregation(query, facts, current_db_id=None):
    query_lower = query.lower()

    is_min = any(w in query_lower for w in [
        "least", "lowest", "smallest", "fewest", "minimum", "worst", "youngest"
    ])
    is_max = any(w in query_lower for w in [
        "most", "largest", "highest", "biggest", "maximum", "max",
        "best", "oldest", "popular"
    ]) and not is_min

    wants_entity = query_lower.startswith((
        "who", "which", "what person", "what city", "what country", "what team"
    ))
    
    # Questions such as "What is the largest X" require a numeric answer rather than an entity.
    wants_number = query_lower.startswith(("what is", "what was", "how many", "how much"))

    numeric_keywords = [
        "attendance", "population", "inhabitants", "yearly", "annual",
        "annually", "visitors", "visitor", "capacity", "seats", "holds",
        "voters", "eligible", "registered", "unemployment", "production",
        "crowd", "attendees", "people", "residents", "citizens"
    ]

    # Branch 1: numeric scan over all facts rather than only top-ranked evidence.
    # This branch mitigates cases where retrieval may miss the relevant numeric fact.
    if is_min or is_max:
        candidates = extract_numeric_candidates(facts, numeric_keywords, query_lower)
        
        if candidates:
            best_val = min(c[0] for c in candidates) if is_min \
                       else max(c[0] for c in candidates)
            tied = [c for c in candidates if c[0] == best_val]
            
            # Tie-breaking with the cross-encoder reranker.
            if len(tied) == 1:
                value, entity, _ = tied[0]
            else:
                cross_scores = reranker.predict([[query, c[2]] for c in tied])
                best_idx = int(np.argmax(cross_scores))
                value, entity, _ = tied[best_idx]
            
            # Decide whether to return a numeric value or an entity.
            if wants_number:
                return str(int(value)) if value == int(value) else str(value)
            if wants_entity and entity:
                return clean_answer(entity)
            # Default behavior: return the numeric value when the question does not explicitly request an entity.
            return str(int(value)) if value == int(value) else str(value)

    # Branch 2: frequency-based aggregation for entity-count patterns.
relevant_facts = hybrid_retrieve_reranked(
        query, facts, current_db_id, top_k=25, rerank_pool=60
    )
    
    from collections import Counter
    entity_counts = Counter()
    for fact in relevant_facts:
        subj = get_person_name_clean(fact) or \
               fact.split(" has ")[0].split(" is ")[0].split(" directed")[0].strip()
        if subj and len(subj) > 2:
            entity_counts[subj] += 1

    if entity_counts and len(entity_counts) > 1:
        best_count = min(entity_counts.values()) if is_min \
                     else max(entity_counts.values())
        tied_entities = [e for e, c in entity_counts.items() if c == best_count]
        sorted_counts = sorted(entity_counts.values(), reverse=True)
        second_count  = sorted_counts[1] if len(sorted_counts) > 1 else None

        if second_count is None or best_count != second_count:
            best_entity = tied_entities[0]
        elif len(tied_entities) <= 5:
            best_entity = rerank_tiebreak(query, tied_entities, relevant_facts)
        else:
            candidates_str = ", ".join(tied_entities[:6])
            context = "\n".join([f"- {f}" for f in relevant_facts])
            confirmation = qwen_generate(
                f"Facts:\n{context}\n\nQuestion: {query}\n\n"
                f"The possible answers are: {candidates_str}\n"
                f"Which one is correct based on the facts?",
                max_new_tokens=10,
                system_prompt="Reply with ONLY the correct answer from the list provided."
            )
            confirmed = clean_answer(confirmation)
            best_entity = next(
                (t for t in tied_entities
                 if confirmed.lower() in t.lower() or t.lower() in confirmed.lower()),
                tied_entities[0]
            )

        if wants_entity:
            return clean_answer(best_entity)
        if "how many" in query_lower:
            return str(best_count)

    # Branch 3: direct Qwen fallback.
    context = "\n".join([f"- {f}" for f in relevant_facts])
    ans = qwen_generate(
        f"Facts:\n{context}\n\nQuestion: {query}",
        max_new_tokens=15,
        system_prompt=(
            "You are a precise data extraction assistant. "
            "Compare all facts carefully. "
            "Reply with ONLY the exact number or entity name, no explanation."
        )
    )
    ans = clean_answer(ans)
    if ans and ans.lower() not in {"unknown", "none", ""}:
        return ans

    # Branch 4: FLAN-T5 fallback.
    prompt = (
        "Answer the question using only the facts.\n"
        "Return only the final exact answer, no explanation.\n\n"
        f"Question: {query}\n\nFacts:\n{context}\n\nAnswer:"
    )
    return clean_answer(flan_extract(prompt))

# Boolean questions: FLAN-T5 component with deterministic checks.
def answer_boolean(query, facts, current_db_id=None):
    query_lower = query.lower()
    relevant_facts = hybrid_retrieve_reranked(query, facts, current_db_id, top_k=8)

    # Gender-specific
    if asks_female(query):
        if any(fact_has_female_marker(f) for f in relevant_facts): return "yes"
        if any(fact_has_male_marker(f) for f in relevant_facts):   return "no"
    if asks_male(query):
        if any(fact_has_male_marker(f) for f in relevant_facts):   return "yes"
        if any(fact_has_female_marker(f) for f in relevant_facts): return "no"

    context = "\n".join(relevant_facts)

    # Membership / part-of
    if has_word(query_lower, ["member"]) or "part of" in query_lower:
        prompt = (
            "Answer the question using only the facts.\n"
            "Return exactly one word: Yes or No.\n\n"
            f"Facts:\n{context}\n\n"
            f"Question: {query}\nAnswer:"
        )
        parsed = parse_yes_no(flan_extract(prompt))
        return parsed if parsed else "no"

    # General FLAN-T5 fallback.
    prompt = (
        "Based only on the facts, answer the question.\n"
        "Return exactly one word: Yes or No.\n"
        "If the facts do not support the statement, answer No.\n\n"
        f"Facts:\n{context}\n\n"
        f"Question: {query}\nAnswer:"
    )
    parsed = parse_yes_no(flan_extract(prompt))
    return parsed if parsed else "no"


# Set operations: gender-based heuristics with a Qwen fallback for generic cases.
def answer_set_ops(query, facts, current_db_id=None):
    query_lower = query.lower()
    if "male" in query_lower and "female" not in query_lower:
        people = set()
        for fact in facts:
            if not is_person_fact(fact): continue
            fl = fact.lower()
            if any(w in fl for w in [" she "," her "," woman ","female"]): continue
            if any(w in fl for w in [" he "," his "," man "," male "]):
                name = get_person_name(fact)
                if name: people.add(name)
        if people: return json.dumps(list(people))
    if "female" in query_lower or "woman" in query_lower:
        people = set()
        for fact in facts:
            if not is_person_fact(fact): continue
            if any(w in fact.lower() for w in [" she "," her "," woman ","female","actress"]):
                name = get_person_name(fact)
                if name: people.add(name)
        if people: return json.dumps(list(people))
    return None  # Remaining cases are processed by the batched Qwen fallback.

print("Answer functions OK")

In [ ]:
print("Precomputing corpus embeddings. This cell must be executed before validation.")
corpus_embeddings = {}
for db_id, facts in tqdm(test_evidence.items()):
    corpus_embeddings[str(db_id)] = retriever.encode(
        facts, batch_size=64, convert_to_numpy=True,
        normalize_embeddings=True, show_progress_bar=False)
print(f"Completed embedding generation for {len(corpus_embeddings)} databases.")

In [ ]:
# Master validation cell for stable query-type-specific evaluation.
# Validation engine.
def evaluate_dev_v5(n=500):
    results = {"Lookup": [], "Boolean": [], "Count": [], "Set ops": [], "Aggregation": []}
    
    for idx, row in tqdm(dev.head(n).iterrows(), total=n):
        qtype = row["query_type"]
        query = row["query"]
        db_id = str(row["db_id"]).split('.')[0]
        
        # Strict data alignment.
        facts = test_evidence[db_id] if db_id in test_evidence else [str(f) for f in row["db_facts"]]
        if not facts:
            results[qtype].append(0.0)
            continue

        try:
            if qtype == "Boolean":       pred = answer_boolean(query, facts, db_id)
            elif qtype == "Count":       pred = answer_count(query, facts, db_id)
            elif qtype == "Aggregation": pred = answer_aggregation(query, facts, db_id)
            elif qtype == "Lookup":      pred = answer_lookup(query, facts, db_id)
            elif qtype == "Set ops":
                pred = answer_set_ops(query, facts, db_id)
                if pred is None:
                    top_facts = hybrid_retrieve_reranked(query, facts, db_id, top_k=20, rerank_pool=50)
                    context   = "\n".join([f"- {f}" for f in top_facts])
                    ans = qwen_generate(f"Facts:\n{context}\n\nQuestion: {query}", max_new_tokens=100,
                                        system_prompt="Reply with ONLY a valid JSON array of strings.")
                    ans = ans.replace("```json","").replace("```","").strip()
                    try:
                        parsed = json.loads(ans)
                        pred = json.dumps(parsed) if isinstance(parsed, list) else "[]"
                    except: pred = "[]"
        except Exception as e:
            pred = "0"

        gold = str(row["gold_answer"]).strip()
        pred = str(pred).strip()

        if qtype == "Boolean":
            correct = pred.lower() == gold.lower()
        elif qtype == "Set ops":
            try:
                gs = set(json.loads(gold))
                ps = set(json.loads(pred)) if pred.startswith("[") else {pred}
                if not gs and not ps: correct = True
                elif not gs or not ps: correct = False
                else:
                    p2 = len(gs&ps)/len(ps); r2 = len(gs&ps)/len(gs)
                    correct = 2*p2*r2/(p2+r2) if (p2+r2)>0 else 0
            except: correct = 0.0
        else:
            correct = 1.0 if pred.lower() == gold.lower() else 0.0
            
        if qtype in results: results[qtype].append(correct)

    print("\n" + "="*45)
    print("VALIDATION REPORT")
    print("="*45)
    scores = []
    for qtype, vals in results.items():
        if vals:
            s = sum(vals)/len(vals); scores.append(s)
            print(f"  {qtype:15s}: {s:.1%}  ({len(vals)} samples)")
    print("-" * 45)
    print(f"  MACRO AVERAGE  : {sum(scores)/len(scores):.1%}")
    print("="*45)

# Run
evaluate_dev_v5(n=500)

In [ ]:
answers_dict = {}
qwen_rows    = []
error_counts = 0

print("Generating test-set answers...")
for _, row in tqdm(test.iterrows(), total=len(test)):
    qtype = router(row["query"])
    db_id = str(row["db_id"])
    try:
        facts = get_facts(row, is_test=True)
        pred  = None
        if qtype == "Boolean":       pred = answer_boolean(row["query"], facts, db_id)
        elif qtype == "Count":       pred = answer_count(row["query"], facts, db_id)
        elif qtype == "Aggregation": pred = answer_aggregation(row["query"], facts, db_id)
        elif qtype == "Lookup":      pred = answer_lookup(row["query"], facts, db_id)
        elif qtype == "Set ops":
            pred = answer_set_ops(row["query"], facts, db_id)
            if pred is None: qwen_rows.append(row); continue
        answers_dict[row["id"]] = str(pred).strip() if pred else "unknown"
    except Exception as e:
        error_counts += 1
        if error_counts <= 3: print(f"Error {row['id']} ({qtype}): {e}")
        answers_dict[row["id"]] = "unknown"

print(f"Completed: {len(answers_dict)} | Errors: {error_counts}")
print(f"Generic Set operation rows assigned to Qwen batch: {len(qwen_rows)}")

# Batched Qwen inference for generic Set operation queries.
batch_size = 8
qwen_df = pd.DataFrame(qwen_rows) if qwen_rows else pd.DataFrame()
if len(qwen_df) > 0:
    tokenizer.padding_side = "left"
    for i in tqdm(range(0, len(qwen_df), batch_size)):
        batch_df = qwen_df.iloc[i:i+batch_size]
        prompts, ids = [], []
        for _, row in batch_df.iterrows():
            ids.append(row["id"])
            try:
                facts     = get_facts(row, is_test=True)
                top_facts = hybrid_retrieve_reranked(row["query"], facts, str(row["db_id"]), top_k=20, rerank_pool=50)
                context   = "\n".join([f"- {f}" for f in top_facts])
            except: context = ""
            messages = [
                {"role":"system","content":"You are a precise data extraction assistant. Reply with ONLY a valid JSON array of strings."},
                {"role":"user","content":f"Facts:\n{context}\n\nQuestion: {row['query']}"}
            ]
            prompts.append(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))
        inputs = tokenizer(prompts, return_tensors="pt", padding=True,
                           truncation=True, max_length=1500).to(llm.device)
        with torch.no_grad():
            outputs = llm.generate(**inputs, max_new_tokens=100, do_sample=False,
                                    pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id)
        for j, out_ids in enumerate(outputs):
            ans = tokenizer.decode(out_ids[inputs.input_ids[j].shape[0]:], skip_special_tokens=True).strip()
            ans = ans.replace("```json","").replace("```","").strip()
            try:
                parsed = json.loads(ans)
                ans = json.dumps(parsed) if isinstance(parsed, list) else "[]"
            except: ans = "[]"
            answers_dict[ids[j]] = ans

submission = pd.DataFrame({"id": test["id"], "answer": [answers_dict.get(i,"unknown") for i in test["id"]]})
submission.to_csv("submission.csv", index=False)
print(f"\nSaved {len(submission)} rows to submission.csv")
print("Unknown values:", (submission["answer"] == "unknown").sum())
display(submission.head(10))